### 4.1 SARIMA — the univariate baseline

$$\varphi(B)\Phi(B^4)(1-B)^d(1-B^4)^D y_t = \theta(B)\Theta(B^4)\,\varepsilon_t$$

The shipped headline order is $(1,0,2)\times(1,0,2,4)$: last quarter's value and shock,
an MA(2) memory, and the same pattern echoed four quarters back — the seasonal term. Zero
exogenous inputs, by design: it exists to answer *how far CPI's own history carries a
forecast*, before any macro predictor is added.

*Source: `src/models/sarima.py`*

#### Explained, step by step

- $B$ is the **backshift operator**: it means "go back one time step." $B\,y_t = y_{t-1}$, and $B^2 y_t = y_{t-2}$. $B^4$ goes back four quarters (one year) — how the model reaches the seasonal pattern.
- $\varphi(B)$ is the **non-seasonal autoregressive (AR) part**, order $p$. With $p=1$ (the first "1" in $(1,0,2)$), this quarter's CPI depends on last quarter's CPI — one lag back.
- $(1-B)^d$ is **differencing**: converting the series into quarter-over-quarter changes, $d$ times, to remove a trend. Here $d=0$: the model works directly on `cpi_yoy` (itself already a year-over-year growth-rate transform), no further differencing.
- $\theta(B)$ is the **non-seasonal moving-average (MA) part**, order $q$. With $q=2$, the model remembers the last two forecast *errors* $\varepsilon_{t-1}, \varepsilon_{t-2}$ — not past CPI values, past *surprises* the model itself made — and uses them to correct the current forecast.
- $\Phi(B^4)$ and $\Theta(B^4)$ repeat the AR/MA logic on lag-4 (one-year-ago) terms — the **seasonal** part, order $(P,D,Q)=(1,0,2)$ at period 4: whatever happened this quarter last year tends to echo this quarter this year.
- $\varepsilon_t$ is white noise — the part of $y_t$ the model can't explain, unpredictable given everything on the left.
- Reading the whole equation left to right: apply the non-seasonal and seasonal AR/differencing filters to this quarter's CPI — that equals the same filters applied to this quarter's random shock. In plain terms: *this quarter's CPI = a weighted mix of last quarter's CPI + last year's same-quarter pattern + memory of recent forecast misses + a fresh unpredictable shock.* Deliberately **zero exogenous inputs** — this is the "how far can inflation's own past carry a forecast, with no macro help at all" baseline that Elastic Net and the Ensemble are judged against.

In [40]:
from src.models.sarima import DEFAULT_ORDER, DEFAULT_SEASONAL_ORDER
print(f"Shipped headline SARIMA order: {DEFAULT_ORDER} x {DEFAULT_SEASONAL_ORDER}")

Shipped headline SARIMA order: (1, 0, 2) x (1, 0, 2, 4)
